In [3]:
import numpy as np
from ase.io import read

In [ ]:
# Get list of unique natoms in xyz file
from ase.io import iread
import numpy as np

frames = list(iread("/home/grethel/dev/quests/examples/gap20/Fullerenes.xyz", format="extxyz"))
natoms = np.array([len(f) for f in frames])

print("Unique natoms:", np.unique(natoms))
print("Counts per natoms:")
for n in np.unique(natoms):
    print(n, np.sum(natoms == n))

In [ ]:
import numpy as np
from pathlib import Path

old_dir = Path("/data/grethel/embeddings/reflect_invert_invariant/npz/test")
new_dir = Path("/data/grethel/embeddings/reflect_invert_invariant/npz/test_fast")

models = [
    "mace_mp_small",
    "mace_mp_large",
    "uma-s-1p1",
    "orb-v3-conservative-inf-omat",
]

for model in models:
    print(f"\nChecking {model}")

    old = np.load(old_dir / f"{model}_Diamond_reflect_invert_invariant.npz")["embeddings"]
    new = np.load(new_dir / f"{model}_Diamond_reflect_invert_invariant.npz")["embeddings"]

    print("Old shape:", old.shape)
    print("New shape:", new.shape)

    # flatten old if needed
    if old.ndim > 2:
        old_flat = old.reshape(old.shape[0], -1)
    else:
        old_flat = old

    print("Old flattened shape:", old_flat.shape)

    # compare
    diff = np.max(np.abs(old_flat - new))
    print("Max absolute difference:", diff)

    print("Allclose:", np.allclose(old_flat, new))


Checking mace_mp_small
Old shape: (5400, 128, 16)
New shape: (5400, 2048)
Old flattened shape: (5400, 2048)
Max absolute difference: 0.0
Allclose: True


In [4]:
# pred_path = "/data/grethel/embeddings/reflect_invert/npz/random/uma-s-1p1_Diamond_reflect_invert.npz"
pred_path = "/data/grethel/embeddings/reflect_invert/npz/uma-s-1p1_Diamond_reflect_invert.npz"

label_path = "/home/grethel/dev/quests/examples/gap20_reflect_invert/Diamond_reflect_invert.xyz"
# label_path = "/home/grethel/dev/quests/examples/gap20/Diamond.xyz"

data = np.load(pred_path, allow_pickle=True)
frames = read(label_path, index=":", format="extxyz")

In [10]:
print(data)
print(data['embeddings'].shape)
print(data['energy'].shape)
print(data['forces'].shape)

NpzFile '/data/grethel/embeddings/reflect_invert/npz/random/uma-s-1p1_Graphene_reflect_invert.npz' with keys: energy, forces, stress, embeddings
(646400, 9, 128)
(3232,)
(646400, 3)


In [5]:
forces_label = [f.arrays["force"] for f in frames]
forces_pred = data['forces']

forces_label = np.concatenate(forces_label, axis=0)
# forces_pred = np.concatenate(forces_pred, axis=0)
print(forces_label.shape)
print(forces_pred.shape)

rmse = np.sqrt(np.mean((forces_pred - forces_label) ** 2))
print("Force RMSE:", rmse)

(43200, 3)
(43200, 3)
Force RMSE: 18.086529537033062


In [6]:
energy_label = np.array([f.calc.results['energy'] for f in frames])
energy_pred = data['energy']
natoms = np.array([len(f) for f in frames])
energy_label_pa = energy_label / natoms
energy_pred_pa  = energy_pred  / natoms
print(energy_label_pa)
print(energy_pred_pa)

mae_mev = np.mean(np.abs(energy_pred_pa - energy_label_pa)) * 1000
rmse_mev = np.sqrt(np.mean((energy_pred_pa - energy_label_pa)**2)) * 1000

dE = energy_pred - energy_label
mae_E  = np.mean(np.abs(dE))
rmse_E = np.sqrt(np.mean(dE**2))

print(f"MAE:  {mae_mev:.3f} meV/atom")
print(f"RMSE: {rmse_mev:.3f} meV/atom")
print(f"System MAE:  {mae_E:.6f} eV/frame")
print(f"System RMSE: {rmse_E:.6f} eV/frame")

[-7.89468362 -7.89468362 -7.89468362 ... -7.81949701 -7.81949701
 -7.81949701]
[-9.04716669 -9.0471655  -9.04716562 ... -4.85787331 -8.98768996
 -8.9875965 ]
MAE:  1620.995 meV/atom
RMSE: 1816.878 meV/atom
System MAE:  38.007657 eV/frame
System RMSE: 52.340725 eV/frame


In [6]:
import json
import numpy as np
from ase.io import read
from pathlib import Path

# -------------------- Paths --------------------
pred_dir = Path("/data/grethel/embeddings/reflect_invert/npz")
label_dir = Path("/home/grethel/dev/quests/examples/gap20_reflect_invert")
out_path = Path("/home/grethel/dev/quests/sweep_results/embeddings/errors.jsonl")

# # ----- Random (UMA) -----
# pred_dir = Path("/data/grethel/embeddings/reflect_invert/npz/random")
# label_dir = Path("/home/grethel/dev/quests/examples/gap20_reflect_invert")
# out_path = Path("/home/grethel/dev/quests/sweep_results/embeddings/random/errors.jsonl")

# # ----- Strain 0.001 (UMA, MACE) -----
# pred_dir = Path("/data/grethel/embeddings/reflect_invert/npz/strain_0.001")
# label_dir = Path("/home/grethel/dev/quests/examples/gap20_reflect_invert")
# out_path = Path("/home/grethel/dev/quests/sweep_results/embeddings/strain_0.001/errors.jsonl")

# # ----- Strain 0.01 (UMA, MACE) -----
# pred_dir = Path("/data/grethel/embeddings/reflect_invert/npz/strain_0.01")
# label_dir = Path("/home/grethel/dev/quests/examples/gap20_reflect_invert")
# out_path = Path("/home/grethel/dev/quests/sweep_results/embeddings/strain_0.01/errors.jsonl")

# # ----- Strain 0.1 (UMA, MACE) -----
# pred_dir = Path("/data/grethel/embeddings/reflect_invert/npz/strain_0.1")
# label_dir = Path("/home/grethel/dev/quests/examples/gap20_reflect_invert")
# out_path = Path("/home/grethel/dev/quests/sweep_results/embeddings/strain_0.1/errors.jsonl")

# -------------------- Config --------------------
datasets = [
    "Graphene",
    "Diamond",
    "Graphite",
    "Nanotubes",
    "Fullerenes",
    "Liquid",
]

models = [
    # "mace_off_small",
    # "mace_off_medium",
    # "mace_off_large",
    "mace_mp_small",
    "mace_mp_medium",
    "mace_mp_large",
    "uma-s-1p1",
    "uma-m-1p1",
    # "eqV2_31M_omat_mp_salex",
    # "eqV2_86M_omat_mp_salex",
    # "eqV2_153M_omat_mp_salex",
    # "eqV2_dens_31M_mp",
    # "eqV2_dens_86M_mp",
    # "eqV2_dens_153M_mp",
    # "orb-v3-conservative-inf-omat",
    # "orb-v3-conservative-20-omat",
    # "orb-v3-conservative-inf-mpa",
    # "orb-v3-conservative-20-mpa",
]

# -------------------- Load existing JSONL --------------------
existing_runs = {}

if out_path.exists():
    with out_path.open("r") as f:
        for line in f:
            if line.strip():
                existing_runs.update(json.loads(line))

print(f"Loaded {len(existing_runs)} existing runs")

# -------------------- Main loop --------------------
new_runs = {}

for model in models:
    for dataset in datasets:
        run_key = f"{model}_{dataset}_reflect_invert"

        if run_key in existing_runs:
            print(f"{run_key} already exists, skipping.")
            continue

        pred_path = pred_dir / f"{run_key}.npz"
        label_path = label_dir / f"{dataset}_reflect_invert.xyz"

        if not pred_path.exists():
            print(f"{run_key}: predictions not found, skipping.")
            continue

        if not label_path.exists():
            print(f"{run_key}: labels not found, skipping.")
            continue

        # ---------- Load data ----------
        data = np.load(pred_path, allow_pickle=True)
        frames = read(label_path, index=":", format="extxyz")

        # ---------- Forces ----------
        forces_label = np.concatenate(
            [f.arrays["force"] for f in frames], axis=0
        )
        forces_pred = data["forces"]

        forces_rmse = np.sqrt(
            np.mean((forces_pred - forces_label) ** 2)
        )

        # ---------- Energy (per frame) ----------
        energy_label = np.array(
            [f.calc.results["energy"] for f in frames]
        )
        energy_pred = data["energy"]

        dE = energy_pred - energy_label
        energy_mae = np.mean(np.abs(dE))
        energy_rmse = np.sqrt(np.mean(dE ** 2))

        print(
            f"{run_key} | "
            f"F_RMSE={forces_rmse:.6e} | "
            f"E_MAE={energy_mae:.6e} | "
            f"E_RMSE={energy_rmse:.6e}"
        )

        new_runs[run_key] = {
            "forces_rmse": float(forces_rmse),
            "energy_mae": float(energy_mae),
            "energy_rmse": float(energy_rmse),
        }

# -------------------- Append to JSONL --------------------
if new_runs:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("a") as f:
        for k, v in new_runs.items():
            json.dump({k: v}, f)
            f.write("\n")

    print(f"Appended {len(new_runs)} new runs to {out_path}")
else:
    print("No new runs to write.")

Loaded 28 existing runs
mace_mp_small_Graphene_reflect_invert already exists, skipping.
mace_mp_small_Diamond_reflect_invert already exists, skipping.
mace_mp_small_Graphite_reflect_invert already exists, skipping.
mace_mp_small_Nanotubes_reflect_invert already exists, skipping.
mace_mp_small_Fullerenes_reflect_invert already exists, skipping.
mace_mp_small_Liquid_reflect_invert | F_RMSE=6.722321e+04 | E_MAE=5.925202e+04 | E_RMSE=3.292378e+05
mace_mp_medium_Graphene_reflect_invert already exists, skipping.
mace_mp_medium_Diamond_reflect_invert already exists, skipping.
mace_mp_medium_Graphite_reflect_invert already exists, skipping.
mace_mp_medium_Nanotubes_reflect_invert already exists, skipping.
mace_mp_medium_Fullerenes_reflect_invert already exists, skipping.
mace_mp_medium_Liquid_reflect_invert | F_RMSE=1.182068e+05 | E_MAE=5.850671e+04 | E_RMSE=2.840016e+05
mace_mp_large_Graphene_reflect_invert already exists, skipping.
mace_mp_large_Diamond_reflect_invert already exists, skippin

In [ ]:
import os
import numpy as np

src_root = "/data/grethel/embeddings/reflect_invert/npz"
dst_root = "/data/grethel/embeddings/reflect_invert/npz_no_embedding"

for root, dirs, files in os.walk(src_root):
    # Mirror directory structure
    rel = os.path.relpath(root, src_root)
    out_dir = os.path.join(dst_root, rel)
    os.makedirs(out_dir, exist_ok=True)

    for f in files:
        if not f.endswith(".npz"):
            continue

        src_path = os.path.join(root, f)
        dst_path = os.path.join(out_dir, f)

        with np.load(src_path) as data:
            # Extract only desired keys
            filtered = {k: data[k] for k in ["energy", "forces", "stress"] if k in data}

        np.savez(dst_path, **filtered)

print("Done.")